# Notebook 4: Terminus temporal graph memory and branch-local recall

This notebook focuses on the persistent-memory layer: filesystem working memory is ephemeral, while canonical and speculative memory live in branch-local Terminus graphs that can be queried independently for historical and temporal reasoning.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


In [ ]:
from src.active_memory.models import WorkingItem
from src.manifold_sidecar import ManifoldRankingService
from src.persistence.pipeline import MutationPipeline
from src.terminus.adapter import TerminusMemoryRepository
from src.terminus.branch_manager import inference_branch_name, session_branch_name

memory_root = REPO_ROOT / "notebooks" / ".demo-memory-04"
shutil.rmtree(memory_root, ignore_errors=True)
repo = TerminusMemoryRepository(url="http://localhost:9999")
pipeline = MutationPipeline(
    memory_root,
    enable_terminus=True,
    terminus_repo=repo,
    manifold_service=ManifoldRankingService(model_id="notebook-ranker-v1", geometry_version="temporal-graph-demo-v1"),
    enable_inference=True,
)

In [ ]:
for session_id, content in [
    (
        "timeline-a",
        "On Monday, Acme Payments rotated the edge certificate for checkout traffic and 401 errors started shortly after.",
    ),
    (
        "timeline-b",
        "On Tuesday, Acme Payments restored the previous certificate bundle and checkout success rates recovered.",
    ),
]:
    pipeline.run(
        WorkingItem(item_type="observation", content=content, session_id=session_id),
        session_id=session_id,
    )

branches = {
    "session_a": session_branch_name("timeline-a"),
    "session_b": session_branch_name("timeline-b"),
    "inference_a": inference_branch_name("timeline-a"),
    "inference_b": inference_branch_name("timeline-b"),
}
show(branches)

In [ ]:
branch_views = {
    branch_name: {
        "claims": repo.query_claims(branch_name),
        "memories": repo.query_memories(branch=branch_name),
        "inference_nodes": repo.query_inference_nodes(branch_name),
        "facet_relations": repo.query_facet_relations(branch_name),
    }
    for branch_name in branches.values()
}
show(branch_views)

In [ ]:
temporal_story = [
    {
        "branch": branch_name,
        "memory_texts": [memory["content"] for memory in branch_views[branch_name]["memories"]],
        "inference_times": [node.get("inferred_at") for node in branch_views[branch_name]["inference_nodes"]],
    }
    for branch_name in [branches["session_a"], branches["session_b"], branches["inference_a"], branches["inference_b"]]
]
show({
    "canonical_vs_speculative": {
        "session_branch_count": sum(len(branch_views[name]["memories"]) for name in [branches["session_a"], branches["session_b"]]),
        "inference_branch_count": sum(len(branch_views[name]["inference_nodes"]) for name in [branches["inference_a"], branches["inference_b"]]),
    },
    "temporal_story": temporal_story,
})

## What this notebook demonstrates

- filesystem writes feeding a canonical persistent memory graph
- separate `session/*` and `inference/*` graph branches
- branch-local recall for historical/temporal inspection
- persistent speculative state without contaminating trusted session memory